## **Project 1 Phase 2**
Rylee Sandifer and Cheney Floyd

**Introduction:**

**Business Problem and What we hope to Achieve:**

The business problem is that many borrowers may not repay their loans and may choose to default, causing the bank to lose money. We hope to achieve a successful model that uses variables including characteristics and statistics of potential borrowers to determine what matters: whether they will default on their loan. The target column tells us whether or not the client paid off the loan or not. A 1 means that the client had payment difficulties, and a 0 means the client did not. We hope to reduce false negative predictions so that we reduce the amount of defaulters approved for loans.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

from sklearn.preprocessing import StandardScaler


**Data Understanding:**

The Home Credit Default Risk Competition includes tables we used in our project. The main table is called application_train.csv and includes information such as characteristics and statistics on individuals in the form of variables like gender, days employed, age, etc. to help predict default. The supplementary tables we joined to the main table include credit_card_balance.csv, previous_application.csv, and instalments_payments.csv. The table credit_card_balance.csv includes information on individuals' monthly credit card habits, such as days past due and outstanding balance. The table previous_application includes information on individuals' loan applications in the past, such as whether an application was approved or refused, the amounts individuals applied for, and the amounts they were approved for. The table installments_payments.csv includes information about whether installment payments were made, such as whether a payment was on time or late. We joined these supplementary tables to the main table to include more information about each individual to more accurately predict default for each individual.

**Data Preparation:**

Read in the main table and three supplementary tables to dataframes:

In [ ]:
df = pd.read_csv("data/application_train.csv")
df2 = pd.read_csv("data/credit_card_balance.csv")
df3=pd.read_csv("data/previous_application.csv")
df4=pd.read_csv("data/installments_payments.csv")


<>:1: SyntaxWarning: invalid escape sequence '\S'
<>:2: SyntaxWarning: invalid escape sequence '\S'
<>:3: SyntaxWarning: invalid escape sequence '\S'
<>:4: SyntaxWarning: invalid escape sequence '\S'
<>:1: SyntaxWarning: invalid escape sequence '\S'
<>:2: SyntaxWarning: invalid escape sequence '\S'
<>:3: SyntaxWarning: invalid escape sequence '\S'
<>:4: SyntaxWarning: invalid escape sequence '\S'
C:\Users\rylee\AppData\Local\Temp\ipykernel_42424\3129510142.py:1: SyntaxWarning: invalid escape sequence '\S'
  df = pd.read_csv("C:/Users/rylee/OneDrive - Centre College of Kentucky\Sophomore College/Spring Term 2026/DSC 340 - Machine Learning/project1/home-credit-default-risk/application_train.csv")
C:\Users\rylee\AppData\Local\Temp\ipykernel_42424\3129510142.py:2: SyntaxWarning: invalid escape sequence '\S'
  df2 = pd.read_csv("C:/Users/rylee/OneDrive - Centre College of Kentucky\Sophomore College/Spring Term 2026/DSC 340 - Machine Learning/project1/home-credit-default-risk/credit_card_bal

Drop columns containing more than 50% missing data:

We determined the percentage of missing data from all columns in the four dataframes. We decided to drop all columns that were missing more than 50% of data in the column. We did this because any column with more than half of its data missing is unstable and isn't a strong basis of prediction. They are missing too much data/information to provide any useful analysis or predictions. If we imputed these missing values, it wouldn't accurately represent the individuals because there is too much data missing to impute or fill in the data, as that would cause bias and disrupt the original distribution.


In [3]:
missing = df.isnull().sum()/len(df)
missing2= df2.isnull().sum()/len(df2)
missing3= df3.isnull().sum()/len(df3)
missing4= df4.isnull().sum()/len(df4)

overfiftymissing = missing[missing > 0.5].index.tolist()
overfiftymissing2=missing2[missing2>0.5].index.tolist()
overfiftymissing3=missing3[missing3>0.5].index.tolist()
overfiftymissing4=missing4[missing4>0.5].index.tolist()

threshold = int(len(df) * 0.5) + 1
threshold2 = int(len(df2) * 0.5) + 1
threshold3 = int(len(df3) * 0.5) + 1
threshold4 = int(len(df4) * 0.5) + 1

df_cleaned = df.dropna(axis=1, thresh=threshold)
df_cleaned2 = df2.dropna(axis=1, thresh=threshold2)
df_cleaned3=df3.dropna(axis=1,thresh=threshold3)
df_cleaned4=df4.dropna(axis=1,thresh=threshold4)


Drop FLAG_DOCUMENT columns:

We decided to drop all columns that contained "FLAG_DOCUMENT". These columns only show whether a client included a certain document, and these columns largely included all zeroes, which doesn't help with prediction and we therefore concluded these columns were not significant. We dropped these columns because they don't add value to our model and dropping them reduces noise in our model.

In [4]:
cols = df_cleaned.columns.tolist()
cols
for col in cols:
    if "FLAG_DOCUMENT" in col:
        df_cleaned = df_cleaned.drop(col, axis=1)

Binary encoding of each categorical column:

We used binary encoding to encode columns including the gender, contract type, car owner, and house owner in order to use them in our logistic regression model. We replaced these following variables with 1s and 0s: CODE_GENDER (M/F), NAME_CONTRACT_TYPE (Cash/Revolving), FLAG_OWN_CAR (Y/N), and FLAG_OWN_REALTY (Y/N). We also dropped the 4 rows that contained "XNA" in the CODE_GENDER column, as they were proportionately insignificant and were prohibiting us from using binary encoding.

In [5]:
df_cleaned = df_cleaned[df_cleaned['CODE_GENDER']!='XNA']
df_cleaned['CODE_GENDER'] = df_cleaned['CODE_GENDER'].map({'M': 1, 'F': 0})
df_cleaned['NAME_CONTRACT_TYPE'] = df_cleaned['NAME_CONTRACT_TYPE'].map({'Cash loans': 1, 'Revolving loans': 0})
df_cleaned['FLAG_OWN_CAR'] = df_cleaned['FLAG_OWN_CAR'].map({'N': 1, 'Y': 0})
df_cleaned['FLAG_OWN_REALTY'] = df_cleaned['FLAG_OWN_REALTY'].map({'N': 1, 'Y': 0})

Adding back EXT_SOURCE_1:

The column EXT_SOURCE_1 was dropped when we dropped any columns with more than 50% of data missing because it has more than 50% values missing in the main table. 

Despite this, we reasoned since EXT_SOURCE_2 and EXT_SOURCE_3 were already strong features in our Phase 1 model, we determined that EXT_SOURCE_1, which is also a credit score from an external source like EXT_SOURCE_2 and EXT_SOURCE_3 and represents the same type of information, was useful to include and impute missing values instead of dropping the column entirely. 

In [6]:
df_cleaned["EXT_SOURCE_1"] = df.loc[df_cleaned.index, "EXT_SOURCE_1"]

Predictive imputation for EXT_SOURCE columns:

EXT_SOURCE_1, EXT_SOURCE_2, and EXT_SOURCE_3 are credit scores from external sources for each individual in the dataframe and they are some of the strongest predictors of default, as high or low credit score is a reflection of how responsible an individual is. A high credit score indicates low risk for lenders, meaning they've paid back lenders in the past, and a low credit score indicates high risk for lenders, meaning they've had issues paying back lenders in the past. All three of these columns contain missing values.

During Phase 1, we replaced the missing values in the EXT_SOURCE columns using median imputation. However, in Phase 2 we determined that this EXT_SOURCE data is MAR (missing at random) because a missing credit score is likely determined by outside observed variables such as income or employment, and certain types of individuals are likely to be missing credit score, but this isn't related to the actual value itself. We used a Random Forest Regressor to predict imputation of missing values in EXT_SOURCE columns to improve our model. For each EXT_SOURCE column, we used rows where a value was known to train a model. Our inputs were all numeric columns, and we used this model to predict the missing values in EXT_SOURCE columns. 

This method provides better estimates for each individual based on other numeric values and variables and better represents the true distribution of data rather than creating a single value to impute for each missing value.

This change produced the largest single improvement in our model, increasing AUC by 0.038.

In [7]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer

def model_impute(df, column):

    known = df[df[column].notna()]
    missing = df[df[column].isna()]

    if missing.shape[0] == 0:
        return df

    X_train = known.drop(column, axis=1)
    y_train = known[column]

    X_missing = missing.drop(column, axis=1)

    # keep only numeric columns
    X_train = X_train.select_dtypes(include=["number"])
    X_missing = X_missing.select_dtypes(include=["number"])

    imputer = SimpleImputer(strategy="median")

    X_train_imp = imputer.fit_transform(X_train)
    X_missing_imp = imputer.transform(X_missing)

    model = RandomForestRegressor(
        n_estimators=200,
        random_state=1,
        n_jobs=-1
    )

    model.fit(X_train_imp, y_train)

    preds = model.predict(X_missing_imp)

    df.loc[df[column].isna(), column] = preds

    return df

In [8]:
df_cleaned = model_impute(df_cleaned, "EXT_SOURCE_1")
df_cleaned = model_impute(df_cleaned, "EXT_SOURCE_2")
df_cleaned = model_impute(df_cleaned, "EXT_SOURCE_3")

Creating EXT_SOURCE summary variable:

We created a variable in our main dataframe that calculated the mean of all EXT_SOURCE columns. The mean value of EXT_SOURCE variables is a single value that demonstrates an individual's overall creditworthiness and whether a lender should trust them to pay them back. We tested this variable by itself and also alongside the other three EXT_SOURCE columns and we found that the AUC improves when we use the three columns in addition to the mean column.

In [9]:
df_cleaned["EXT_SOURCE_MEAN"] = df_cleaned[
    ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
].mean(axis=1)

Feature Engineering - financial ratio features:

We engineered four features through ratios that demonstrate how existing variables relate to one another. These added features summarize existing variables and help us make sense out of them.

- CREDIT_TO_INCOME: total loan amount divided by annual income — measures how large the loan is compared to what the borrower earns (taking out a loan worth years worth of salary is risker than taking out a loan worth 6 months of salary)
- ANNUITY_TO_INCOME: monthly loan payment divided by annual income — measures percentage of the borrower's income that goes toward their monthly loan payment (if income going straight to loan payments, they will fall behind quicker)
- GOODS_TO_CREDIT: goods price divided by credit amount — measures how much of the loan is actually going toward purchasing goods versus other costs (a loan with a specific purpose is more predictable than a loan with unexplained borrowing)
- EMPLOYED_TO_AGE: days employed divided by days of birth — measures employment stability relative to age (a young person employed for many years is a different risk than an older person with the same employment length)

Each of these features is cleaned from division by zero (infinite values) and any existing missing values are imputed with the median of the column. The missing values in these columns are MCAR (missing completely at random) because they result from division of zero. The missing values don't have to do with the borrower's risk level. For MCAR, median is appropriate and doesn't add bias to the distribution.

Adding these features through feature engineering improved our AUC score overall by around ~0.013.

In [10]:
df_cleaned["CREDIT_TO_INCOME"] = (
    df_cleaned["AMT_CREDIT"] / df_cleaned["AMT_INCOME_TOTAL"]
)
df_cleaned["CREDIT_TO_INCOME"] = df_cleaned["CREDIT_TO_INCOME"].replace(
    [np.inf, -np.inf], np.nan
).fillna(df_cleaned["CREDIT_TO_INCOME"].median())



df_cleaned["ANNUITY_TO_INCOME"] = (
    df_cleaned["AMT_ANNUITY"] / df_cleaned["AMT_INCOME_TOTAL"]
)
df_cleaned["ANNUITY_TO_INCOME"] = df_cleaned["ANNUITY_TO_INCOME"].replace(
    [np.inf, -np.inf], np.nan
).fillna(df_cleaned["ANNUITY_TO_INCOME"].median())



df_cleaned["GOODS_TO_CREDIT"] = (
    df_cleaned["AMT_GOODS_PRICE"] / df_cleaned["AMT_CREDIT"]
)
df_cleaned["GOODS_TO_CREDIT"] = df_cleaned["GOODS_TO_CREDIT"].replace(
    [np.inf, -np.inf], np.nan
).fillna(df_cleaned["GOODS_TO_CREDIT"].median())



df_cleaned["EMPLOYED_TO_AGE"] = (
    df_cleaned["DAYS_EMPLOYED"] / df_cleaned["DAYS_BIRTH"]
)
df_cleaned["EMPLOYED_TO_AGE"] = df_cleaned["EMPLOYED_TO_AGE"].replace(
    [np.inf, -np.inf], np.nan
).fillna(df_cleaned["EMPLOYED_TO_AGE"].median())

Aggregating supplementary tables:

Every supplementary table has multiple rows for each individual. For example, one borrower has multiple credit card statements, previous applications, or installment payments. Our main table has one row for each borrower, so we aggregated each of the supplementary tables into one row for each borrower through summary statistics before we joined tables.

The value counts show that majority of borrowers have multiple rows in these supplementary tables, and this confirms that we need to aggregate before we join tables.

In [11]:
df_cleaned2['SK_ID_CURR'].value_counts().head()
df_cleaned3['SK_ID_CURR'].value_counts().head()
df_cleaned4['SK_ID_CURR'].value_counts().head()

SK_ID_CURR
145728    372
296205    350
453103    347
189699    344
186851    337
Name: count, dtype: int64

Feature engineering from credit card balance data:

We created two ratios based on the credit card balance table before we aggregated the data. We created a variable called credit_utilization which is balance divided by credit limit, and a higher ratio shows that the borrower is struggling more. We also added payment_ratio which is the actual payment divided by minimum required and it shows whether a borrower is meeting their minimum payments, as any number below 1 means they are not meeting their payments.

Then, we aggregated the data to ensure there was one row for each borrower with no duplicates. We chose to focus on variables that reflect behavior of the borrowers. We focused on highest number of days past due, average days past due for definite defaults, maximum credit utilization, average payment ratio, total drawings, and average balance. These variables depict the worst habits and behaviors of a borrower throughout all credit card months. An individual that maxes out their credit card and fails to meet minimum payments demonstrates financial struggle and this increases risk of defaulting on a loan.

In [12]:
df_cleaned2["credit_utilization"] = (
    df_cleaned2["AMT_BALANCE"] /
    df_cleaned2["AMT_CREDIT_LIMIT_ACTUAL"]
)

df_cleaned2["payment_ratio"] = (
    df_cleaned2["AMT_PAYMENT_CURRENT"] /
    df_cleaned2["AMT_INST_MIN_REGULARITY"]
)

import numpy as np

df_cleaned2["credit_utilization"] = df_cleaned2["credit_utilization"].replace([np.inf, -np.inf], np.nan)
df_cleaned2["payment_ratio"] = df_cleaned2["payment_ratio"].replace([np.inf, -np.inf], np.nan)

credit_features = df_cleaned2.groupby("SK_ID_CURR").agg({
    "SK_DPD": ["max"],
    "SK_DPD_DEF": ["mean"],
    "credit_utilization": ["max"],
    "payment_ratio": ["mean"],
    "AMT_DRAWINGS_CURRENT": ["sum"],
    "AMT_BALANCE": ["mean"]
})

credit_features.columns = [
    "SK_DPD_max",
    "SK_DPD_DEF_mean",
    "credit_utilization_max",
    "payment_ratio_mean",
    "AMT_DRAWINGS_CURRENT_sum",
    "AMT_BALANCE_mean"
]

credit_features = credit_features.reset_index()

df_cleaned2.columns = ["_".join(col) for col in df_cleaned2.columns]
df_cleaned2 = df_cleaned2.reset_index()

Feature engineering from previous applications:

We created a variable called REFUSED from the previous application table. This shows if a previous application was refused by a lender. This creates a simple value to determine whether someone has been turned down before.

We then aggregated the data to one row for each borrower. We focused on summary statistics for credit amounts, annuities, payment counts, and decision timing, and we focused a lot on the predictor of the sum of refusals which reflects that those who have a higher number here have been turned down many times and have higher risk of default.

In [13]:
df_cleaned3["REFUSED"] = (df_cleaned3["NAME_CONTRACT_STATUS"] == "Refused").astype(int)

previous_features = df_cleaned3.groupby("SK_ID_CURR").agg({
    "AMT_CREDIT": ["mean", "max"],
    "AMT_APPLICATION": ["mean"],
    "CNT_PAYMENT": ["mean"],
    "AMT_ANNUITY": ["mean"],
    "DAYS_DECISION": ["mean"],
    "REFUSED":["sum"]
})

previous_features.columns = [
    "AMT_CREDIT_mean",
    "AMT_CREDIT_max",
    "AMT_APPLICATION_mean",
    "CNT_PAYMENT_mean",
    "AMT_ANNUITY_mean",
    "DAYS_DECISION_mean",
    "REFUSED_sum"
]

previous_features = previous_features.reset_index()

Feature engineering from installment payments:

We created a variable called DAYS_LATE from the installments payments table. This is the difference of the actual and scheduled payment dates. If this value is positive, the payment was late, and if it's negative then the payment was on time. We aggregated the data to again have one row for each borrower, and we focused on the mean and maximum lateness for each borrower, and we also focused on summary statistics for installment amounts and payment amounts. This is important because an individual who is always late on their payments has higher risk than someone who is never late on their payments.

In [14]:
df_cleaned4["DAYS_LATE"] = (
    df_cleaned4["DAYS_ENTRY_PAYMENT"].fillna(0) - 
    df_cleaned4["DAYS_INSTALMENT"].fillna(0)
)
df_cleaned4["DAYS_LATE"] = (
    df_cleaned4["DAYS_LATE"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
    .astype(int)
)

installment_features = df_cleaned4.groupby("SK_ID_CURR").agg({
    "NUM_INSTALMENT_NUMBER": ["max"],
    "NUM_INSTALMENT_VERSION": ["mean"],
    "DAYS_INSTALMENT": ["mean"],
    "DAYS_ENTRY_PAYMENT": ["mean"],
    "AMT_INSTALMENT": ["mean"],
    "AMT_PAYMENT": ["mean"],
    "DAYS_LATE": ["mean", "max"]
})

installment_features.columns = [
    "NUM_INSTALMENT_NUMBER_max",
    "NUM_INSTALMENT_VERSION_mean",
    "DAYS_INSTALMENT_mean",
    "DAYS_ENTRY_PAYMENT_mean",
    "AMT_INSTALMENT_mean",
    "AMT_PAYMENT_mean",
    "DAYS_LATE_mean",
    "DAYS_LATE_max"
]

installment_features = installment_features.reset_index()

Join all tables:

After aggregating the supplementary tables, we joined them to the main dataframe on SK_ID_CURR with left joins. We used left joins to make sure that we keep all of the individuals from the main dataframe, even if they don't have any credit card history, previous applications, or previous installment payments. We take care of these missing values in our next step.

In [15]:
df_joined = df_cleaned.merge(credit_features, on="SK_ID_CURR", how="left")
df_joined = df_joined.merge(previous_features, on="SK_ID_CURR", how="left")
df_joined=df_joined.merge(installment_features,on="SK_ID_CURR", how="left")

Fill remaining missing values:

After we joined tables, it resulted with individuals with missing values in the supplementary tables. This means that they don't have any history in the supplementary tables, and therefore we fill these missing values with 0. We do this because a borrower who has no history can be accurately represented with 0 in these columns. If they have no credit card history, no previous applications, and no installment payments, then these values are 0 - there is a zero balance, zero payments, etc.

In [16]:
df_joined = df_joined.fillna(0)

Feature selection:

The features present in our final model are drawn from the main table and three supplementary tables. We included the three EXT_SOURCE features and the calculated mean. We included features such as age, length of employment, registration, risk indicators based on region, engineered ratio features, and aggregated behavioral features from credit card, previous application, and installment payment history. We chose to include these features because together, they reflect the borrower's behaviors, how much they borrow compared to their financial situation, and how they've dealt with credit and payment time previously, and these are all helpful and predictive to determine whether they will default. We excluded columns found to be insignificant in earlier testing (FLOORSMAX_AVG, FLOORSMAX_MEDI, FLOORSMAX_MODE) as removing them had negligible impact on AUC.

In [17]:
full_keep=["TARGET",
      "EXT_SOURCE_1",
      "EXT_SOURCE_2",
      "EXT_SOURCE_3",
      "EXT_SOURCE_MEAN",
      "DAYS_BIRTH",
      "DAYS_ID_PUBLISH",
      "DAYS_REGISTRATION",
      "DAYS_EMPLOYED",
      "FLAG_EMP_PHONE",
      "REGION_RATING_CLIENT",
      "REGION_RATING_CLIENT_W_CITY",
      "REGION_POPULATION_RELATIVE",
      "REG_CITY_NOT_WORK_CITY",
      "REG_CITY_NOT_LIVE_CITY",
      "CREDIT_TO_INCOME",
      "ANNUITY_TO_INCOME",
      "GOODS_TO_CREDIT",
      "EMPLOYED_TO_AGE",
      "SK_DPD_max",
      "SK_DPD_DEF_mean",
      "credit_utilization_max",
      "payment_ratio_mean",
      "AMT_DRAWINGS_CURRENT_sum",
      "AMT_BALANCE_mean",
      "AMT_CREDIT_mean",
      "AMT_CREDIT_max",
      "AMT_APPLICATION_mean",
      "CNT_PAYMENT_mean",
      "AMT_ANNUITY_mean",
      "DAYS_DECISION_mean",
      "REFUSED_sum",
      "NUM_INSTALMENT_NUMBER_max",
      "NUM_INSTALMENT_VERSION_mean",
      "DAYS_INSTALMENT_mean",
      "DAYS_ENTRY_PAYMENT_mean",
      "AMT_INSTALMENT_mean",
      "AMT_PAYMENT_mean",
      "DAYS_LATE_mean",
      "DAYS_LATE_max"]
df_joined=df_joined[full_keep]

**Model and Evaluation:**

We built our logistic regression model through sklearn. We first created our X and y sets by dropping the target column from our joined data frame and separating into X (features) and y (target). The target column tells us whether or not the client paid off the loan or not. A 1 means that the client had payment difficulties, and a 0 means the client did not.

We performed a 70/30 train-test split with our data and stratified on y to keep the datasets proportionate. This is to keep the class ratio on both sets. We then scaled all of the data features with StandardScaler after the split. We only fit on the training set and transformed on both to prevent data leakage. For our model we used class_weight="balanced" to deal with the class imbalance that we see in our data, which addresses the 92/8 class imbalance and allows the model to more heavily weight defaulters while training.

We used C=3 for regularization strength and tested multiple values of C throughout our model building process, such as 0.01, 0.1, 1, and 3. We found that the effect on AUC was low, with changes of less than 0.003, which showed that regularization tuning wasn't a very significant aspect for this model.

We decided to not use the .5 classification threshold and looked at a precision-recall curve, and we instead used a threshold of 0.25. This threshold results in a recall of 0.94 on the default class, which means that our model correctly identifies 94% of borrowers that will have difficulties with payments. This lower threshold focuses on recall and trades precision for recall, which follows our business goal of reducing false negatives and catching the most defaulters that we can.

In [18]:
X = df_joined.drop('TARGET', axis=1)
y = df_joined['TARGET']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1, stratify=y)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(solver='lbfgs', max_iter=1000, class_weight="balanced", C=3) 
model.fit(X_train_scaled, y_train)

predictions = model.predict(X_test_scaled)
probabilities = model.predict_proba(X_test_scaled)[:,1]

predictions = (probabilities > 0.25).astype(int)

print(confusion_matrix(y_test, predictions))
print(classification_report(y_test, predictions))

print(roc_auc_score(y_test, probabilities))

[[28568 56237]
 [  456  6992]]
              precision    recall  f1-score   support

           0       0.98      0.34      0.50     84805
           1       0.11      0.94      0.20      7448

    accuracy                           0.39     92253
   macro avg       0.55      0.64      0.35     92253
weighted avg       0.91      0.39      0.48     92253

0.7870752220406313


Threshold selection using precision-recall curve:

We used the precision-recall curve to determine the tradeoff between recall and precision at differing thresholds. We ended up choosing a threshold of 0.25 that results in a recall of 0.94 and catches 94% of defaulters with a precision of 0.11, which means 11% of applicants that we flag as likely defaulters are actually defaulters. This tradeoff between precision and recall is intentional because our business goal in this situation is to reduce the number of defaulters who we approve for loans, so in this case recall is the most important metric to focus on and take into account.

In [19]:
from sklearn.metrics import precision_recall_curve

precision_vals, recall_vals, thresholds = precision_recall_curve(y_test, probabilities)

for target in [0.88, 0.90, 0.92, 0.94, 0.95, 0.96]:
    idx = (recall_vals >= target).nonzero()[0][-1]
    print(f"Recall={target}: threshold={thresholds[idx]:.3f}, precision={precision_vals[idx]:.3f}")

Recall=0.88: threshold=0.335, precision=0.131
Recall=0.9: threshold=0.309, precision=0.124
Recall=0.92: threshold=0.284, precision=0.118
Recall=0.94: threshold=0.249, precision=0.110
Recall=0.95: threshold=0.229, precision=0.106
Recall=0.96: threshold=0.210, precision=0.102


**Conclusion and Next Steps:**

**Reflection on our Results:**

During Phase 2, we made improvements to our Phase 1 model to improve our results, specifically focusing on recall for defaulters. We joined supplementary tables credit_card_balance.csv, previous_application.csv, and instalments_payments.csv and we used feature engineering to create new features to summarize behavioral features in each table. For the EXT_SOURCE columns, we implemented Random Forest predictive imputation instead of using median imputation, and this was one of our largest improvements to our model in Phase 2. We also added EXT_SOURCE_1 and EXT_SOURCE mean features, and we used feature engineering to create ratio features from supplementary tables. We also chose our threshold of 0.25 through the precision-recall curve.

Our final model results in an AUC of 0.7871 and a recall of 0.94 on the default class, which means it correctly identifies 94% of borrowers who will default on their loan.

Our model is better than the dummy classifier because the most important metric to focus on in this business problem is the recall for defaulters. This is because it's a much higher cost to the business for our model to miss false negatives. We focused our model on improving recall, which is .94, reflecting that our model catches 94% of defaulters. The recall of defaulters of the dummy classifier is 0, and our model's is .94, which is much better than the dummy. Additionally, the precision of defaulters of the dummy classifier is 0, and our precision of defaulters is 0.11, which means of all applicants that we flag as likely to default, 11% actually do default, which means we reject some good applicants, but our goal is to minimize loan defaults, so we accept this tradeoff and it's intended in this model. Additionally, the dummy classifier's AUC score is 0.50, and our model's AUC score is .7871, which exceeds the dummy classifier. The dummy classifier has an accuracy of .92 while our model has an accuracy of .39, but accuracy is misleading here due to class imbalance because if a model approves every loan applicant, it's useless to the bank. We are focused on the recall because that is what matters most for the bank in this business problem. 

**AI Reflection:**

Throughout Phase 1, we used ChatGPT to help us with code and ideas for our current model. It was helpful in the logistic regression code, but not as helpful with the imputation methods because it didn't consider MAR, MNAR, and MCAR, so we had to revise our imputation methods to try to improve our missing data and improve our model. Overall, AI was helpful in getting started with our model and very helpful in determining which columns were likely to be more significant than others.

Throughout Phase 2, we used Claude to help us debug code, get ideas for feature engineering, and determine which model version was better throughout our process. 

Claude helped us debug, recommended ratio features and to replace median imputation with Random Forest prediction imputation for the EXT_SOURCE columns, and helped us choose the most effective threshold. We used Claude to help us get ideas for code and used many options Claude gave us to try with our model and see which options improved our model.

**References and Acknowledgements:**

We received help from Dr. Allen in office hours regarding data preparation techniques for data cleaning. We also referred to the SciKit Learn documentation for logistic regression models.